# CS-13410 Semester Project — Part 1: Data Preparation & Leakage-Free Pipeline
**Project:** Early prediction of student disengagement (withdrawal) — built for the Study-Sync learning platform
**Course:** Introduction to Machine Learning, Fall 2026, The University of Lahore
**Group:** *(add names and roll numbers)*  |  **Repository:** *(add GitHub link)*

> **Attribution note (academic integrity):** the starting structure of this notebook was drafted with help from an AI assistant (Claude, Anthropic). Our group reviewed, ran and adapted it. *(Edit this line to describe what your group changed.)*

**Dataset:** Open University Learning Analytics Dataset (OULAD), CC-BY 4.0. Kuzilek, J., Hlosta, M., & Zdrahal, Z. (2017). Open University Learning Analytics dataset. *Scientific Data, 4*, 170171. https://doi.org/10.1038/sdata.2017.171

## 1. Problem framing

| Item | Decision |
|---|---|
| **Business question** | Can we warn a learner early that they are likely to stop engaging, so Study-Sync can send a reminder, streak nudge or partner prompt *before* they drop out? |
| **Unit of analysis** | One student in one module-presentation (one row per `code_module` + `code_presentation` + `id_student`) |
| **Prediction target** | `withdrew` = 1 if the student's final result is *Withdrawn*, else 0 |
| **Task type** | Supervised **binary classification** |
| **Prediction time (cutoff)** | Only data observed up to day `CUTOFF_DAY` of the course is used as input. Anything after the cutoff is off-limits. |
| **Population** | Students still registered at the cutoff (students who already unregistered are not "at risk" any more) |
| **Success metrics** | Primary: **PR-AUC (average precision)** and **F1 of the withdrawing class**, because the classes are imbalanced and we care about catching the at-risk class. Secondary: ROC-AUC, recall at a fixed precision. Accuracy alone is not used. |
| **Why OULAD suits Study-Sync** | Real learner behaviour (daily click activity, assessment submissions) plus demographics, with more than 1,000 rows, numeric and categorical features, and an explicit withdrawal outcome. Study-Sync's own engagement signals (sessions, streaks, quiz results) map directly onto these feature types. |

**Main leakage risks we control for:** (1) using `final_result` or `date_unregistration` as features, (2) using clicks or assessments *after* the cutoff, (3) fitting imputers/scalers/encoders on test data, (4) the same student appearing in both train and test.

In [ ]:
!wget -O oulad.zip "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"
!mkdir -p oulad
!unzip -o oulad.zip -d oulad
!ls oulad

In [ ]:
# ---- Configuration ----
from pathlib import Path

DATA_DIR = Path("oulad")   # put the OULAD CSV files here (see section 2)
CUTOFF_DAY = 60                 # prediction time, in days relative to module start. Try 30 / 60 / 90 later.
RANDOM_STATE = 42
TEST_SIZE = 0.20

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_STATE)

## 2. Dataset: where to get it and how to load it
1. Download the OULAD zip from the official page: https://analyse.kmi.open.ac.uk/open_dataset (license: CC-BY 4.0, so cite it as above).
2. Extract the CSVs into `data/oulad/`. You need: `studentInfo.csv`, `studentRegistration.csv`, `studentVle.csv`, `studentAssessment.csv`, `assessments.csv`.
3. `studentVle.csv` is large (hundreds of MB), so the code below reads it in chunks and keeps only rows up to the cutoff.

**Do not commit the CSVs to GitHub** (size). Add `data/` to `.gitignore` and describe the download step in the README instead.

In [ ]:
student_info = pd.read_csv(DATA_DIR / "studentInfo.csv")
registration = pd.read_csv(DATA_DIR / "studentRegistration.csv")
assessments  = pd.read_csv(DATA_DIR / "assessments.csv")
student_assm = pd.read_csv(DATA_DIR / "studentAssessment.csv")

KEYS = ["code_module", "code_presentation", "id_student"]

# studentVle: read in chunks and keep only activity up to the cutoff (no future information ever enters memory)
vle_chunks = []
for chunk in pd.read_csv(DATA_DIR / "studentVle.csv", chunksize=1_000_000,
                         dtype={"id_student": "int32", "id_site": "int32", "date": "int16", "sum_click": "int32"}):
    vle_chunks.append(chunk[chunk["date"] <= CUTOFF_DAY])
student_vle = pd.concat(vle_chunks, ignore_index=True)
del vle_chunks

print("studentInfo:", student_info.shape, "| registration:", registration.shape,
      "| VLE rows kept (<= cutoff):", student_vle.shape, "| student_assessment:", student_assm.shape)
student_info.head()

### 2.1 Data cleaning note
The UCI copy of OULAD stores missing values as text ("?"), so some numeric columns load as strings. The cell below converts "?" to proper missing values (NaN) and casts the date, score and count columns to numbers. This must happen before any filtering or feature construction.

In [ ]:
for t in (student_info, registration, assessments, student_assm):
    t.replace(["?", " ", ""], np.nan, inplace=True)

for col in ["date_registration", "date_unregistration"]:
    registration[col] = pd.to_numeric(registration[col], errors="coerce")

assessments["date"] = pd.to_numeric(assessments["date"], errors="coerce")

for col in ["date_submitted", "score", "is_banked"]:
    student_assm[col] = pd.to_numeric(student_assm[col], errors="coerce")

for col in ["num_of_prev_attempts", "studied_credits"]:
    student_info[col] = pd.to_numeric(student_info[col], errors="coerce")

print(registration[["date_registration", "date_unregistration"]].dtypes)

## 3. Build the cohort and the target
We keep only students who are **still registered at the cutoff** (not unregistered before it, and registered on or before it). The target comes from `final_result`, which is used *only* to build the label and is never a feature.

In [ ]:
base = student_info.merge(registration, on=KEYS, how="left")

still_registered = (base["date_unregistration"].isna() | (base["date_unregistration"] > CUTOFF_DAY))
registered_by_cutoff = base["date_registration"].isna() | (base["date_registration"] <= CUTOFF_DAY)
n_before = len(base)
base = base[still_registered & registered_by_cutoff].copy()
print(f"Rows before cohort filter: {n_before:,} | after: {len(base):,} "
      f"(removed {n_before - len(base):,} who had already left or not yet registered at day {CUTOFF_DAY})")

base["withdrew"] = (base["final_result"] == "Withdrawn").astype(int)
print("\nTarget distribution (share):")
print(base["withdrew"].value_counts(normalize=True).rename({0: "stayed (0)", 1: "withdrew (1)"}).round(3))

## 4. Feature construction (cutoff-safe)
All features below are computed only from data with `date <= CUTOFF_DAY`.

| Feature | Meaning |
|---|---|
| `total_clicks`, `active_days`, `n_sites` | Overall VLE engagement up to the cutoff |
| `clicks_last14` , `recent_share` | Activity in the final 14 days before the cutoff, and its share of all clicks (a drop-off signal) |
| `days_since_last_active` | Days between last activity and the cutoff (large = already disengaging) |
| `never_active` | Flag: no VLE activity recorded by the cutoff |
| `n_submitted`, `n_due`, `submission_rate`, `mean_score` | Assessments submitted/due by the cutoff and the average score of those submitted |
| demographics / prior study | `gender`, `region`, `highest_education`, `imd_band`, `age_band`, `num_of_prev_attempts`, `studied_credits`, `disability`, `code_module`, `date_registration` |

Deterministic fills (e.g. 0 clicks when a student has no VLE record) are safe to do here because they use no statistics from the data. Statistical imputation is left to the pipeline.

In [ ]:
# --- VLE engagement features ---
g = student_vle.groupby(KEYS)
vle_feat = pd.DataFrame({
    "total_clicks": g["sum_click"].sum(),
    "active_days": g["date"].nunique(),
    "n_sites": g["id_site"].nunique(),
    "last_active_day": g["date"].max(),
})
recent = student_vle[student_vle["date"] > CUTOFF_DAY - 14].groupby(KEYS)["sum_click"].sum().rename("clicks_last14")
vle_feat = vle_feat.join(recent).reset_index()

# --- Assessment features (only submissions up to the cutoff) ---
sa = student_assm.merge(assessments[["id_assessment", "code_module", "code_presentation", "date"]],
                        on="id_assessment", how="left")
sa = sa[sa["date_submitted"] <= CUTOFF_DAY]
assm_feat = (sa.groupby(KEYS)
               .agg(n_submitted=("id_assessment", "nunique"), mean_score=("score", "mean"))
               .reset_index())

# number of assessments that were due by the cutoff, per module-presentation
due = (assessments[assessments["date"] <= CUTOFF_DAY]
       .groupby(["code_module", "code_presentation"]).size().rename("n_due").reset_index())

# --- Assemble the modelling table ---
df = (base.merge(vle_feat, on=KEYS, how="left")
          .merge(assm_feat, on=KEYS, how="left")
          .merge(due, on=["code_module", "code_presentation"], how="left"))

# deterministic fills (no data statistics involved)
for c in ["total_clicks", "active_days", "n_sites", "clicks_last14", "n_submitted", "n_due"]:
    df[c] = df[c].fillna(0)
df["never_active"] = df["last_active_day"].isna().astype(int)
df["days_since_last_active"] = (CUTOFF_DAY - df["last_active_day"]).fillna(CUTOFF_DAY + 1)
df["recent_share"] = df["clicks_last14"] / (df["total_clicks"] + 1)
df["submission_rate"] = np.where(df["n_due"] > 0, df["n_submitted"] / df["n_due"], np.nan)
df["imd_band"] = df["imd_band"].astype("string").str.replace("%", "", regex=False)

print(df.shape)
df.head()

## 5. Data quality checks and the final feature list
We separate the **target**, the **identifiers/grouping columns** and the **leakage columns** (excluded on purpose) from the model inputs.

In [ ]:
TARGET = "withdrew"
GROUP_COL = "id_student"                       # used for splitting only, never as a feature
LEAKY_OR_ID = ["final_result", "date_unregistration", "last_active_day", "code_presentation", "id_student"]

ORDINAL_COLS = ["imd_band", "age_band", "highest_education"]
NOMINAL_COLS = ["gender", "region", "disability", "code_module"]
SKEWED_NUM   = ["total_clicks", "clicks_last14", "active_days", "n_sites", "days_since_last_active"]
OTHER_NUM    = ["recent_share", "never_active", "n_submitted", "n_due", "submission_rate", "mean_score",
                "num_of_prev_attempts", "studied_credits", "date_registration"]

FEATURES = ORDINAL_COLS + NOMINAL_COLS + SKEWED_NUM + OTHER_NUM
assert not set(FEATURES) & set(LEAKY_OR_ID + [TARGET]), "A leaky column slipped into the feature list!"

X = df[FEATURES]
y = df[TARGET]
groups = df[GROUP_COL]

print("Rows:", len(df), "| Features:", len(FEATURES), "| Duplicate student-module-presentation rows:",
      df.duplicated(KEYS).sum())
print("\nMissing values per feature (count):")
print(X.isna().sum()[X.isna().sum() > 0].sort_values(ascending=False))

## 6. Train/test split (before any fitting)
A student can appear in more than one module-presentation, so we split **by student** (`GroupShuffleSplit`). Otherwise the same person could sit in both train and test and inflate the scores. The test set stays untouched until the final evaluation in Part 3.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])), "Student overlap between train and test!"
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("Withdrawal rate  train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

## 7. Exploratory data analysis (training set only)
EDA uses `X_train` only, so no insight is drawn from the test set.

In [ ]:
eda = X_train.copy()
eda[TARGET] = y_train.values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Viz 1: class balance
counts = eda[TARGET].value_counts().sort_index()
axes[0].bar(["Stayed (0)", "Withdrew (1)"], counts.values, color=["#4C72B0", "#C44E52"])
axes[0].set_title("Class balance in the training set")
axes[0].set_ylabel("Students")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}\n({v/len(eda):.1%})", ha="center", va="bottom")

# Viz 2: engagement distribution by outcome (log scale)
sns.boxplot(data=eda, x=TARGET, y=np.log1p(eda["total_clicks"]), ax=axes[1], palette=["#4C72B0", "#C44E52"], hue=TARGET, legend=False)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["Stayed", "Withdrew"])
axes[1].set_title(f"VLE clicks up to day {CUTOFF_DAY} (log scale) by outcome")
axes[1].set_ylabel("log(1 + total_clicks)")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Viz 3: withdrawal rate by days since last activity
bins = [-1, 0, 3, 7, 14, 30, CUTOFF_DAY + 1]
labels = ["0 (today)", "1-3", "4-7", "8-14", "15-30", "30+/never"]
eda["inactive_bin"] = pd.cut(eda["days_since_last_active"], bins=bins, labels=labels)
rate = eda.groupby("inactive_bin", observed=True)[TARGET].agg(["mean", "size"])
axes[0].bar(rate.index.astype(str), rate["mean"], color="#C44E52")
axes[0].set_title("Withdrawal rate by days since last VLE activity")
axes[0].set_ylabel("Share who withdrew"); axes[0].set_xlabel("Days since last activity at cutoff")

# Viz 4: withdrawal rate by highest education
order = ["No Formal quals", "Lower Than A Level", "A Level or Equivalent", "HE Qualification", "Post Graduate Qualification"]
r2 = eda.groupby("highest_education")[TARGET].mean().reindex(order)
axes[1].barh(order, r2.values, color="#4C72B0")
axes[1].set_title("Withdrawal rate by highest education")
axes[1].set_xlabel("Share who withdrew")
plt.tight_layout(); plt.show()

In [ ]:
# Viz 5: correlation among numeric features (helps spot redundant features)
num_cols = SKEWED_NUM + OTHER_NUM
corr = eda[num_cols + [TARGET]].corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation matrix (training set)")
plt.tight_layout(); plt.show()

**EDA notes — write your own findings here after running the cells** (use the real plots and numbers from your run): class imbalance, which engagement features separate the two groups, which pairs are highly correlated (e.g. `total_clicks` vs `active_days`), and any demographic pattern worth treating with care in the fairness discussion in Part 4.

## 8. Leakage-free preprocessing pipeline
Everything that learns from data (imputation values, scaling statistics, winsorising limits, encoder categories, variance filter) lives **inside** a scikit-learn `Pipeline`/`ColumnTransformer`. It is fitted on training data only and merely *applied* to validation/test data.

| Problem | Handling | Where |
|---|---|---|
| Missing values | Median (numeric, with missing-indicator) / most-frequent (categorical) imputation | fitted on train only |
| Outliers | Winsorise heavy-tailed engagement counts at the 1st/99th percentile, then `log1p` | limits learned on train only |
| Scaling | `StandardScaler` | fitted on train only |
| Encoding | Explicit ordered `OrdinalEncoder` for ordered bands; `OneHotEncoder` for nominal columns | unknown categories at test time are handled safely |
| Feature selection | `VarianceThreshold` removes constant features | fitted on train only |
| Class imbalance | `class_weight="balanced"` in the model plus stratified/grouped CV and PR-AUC/F1 metrics | no resampling of test data |

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    # Clip each column to the [lower, upper] quantiles learned from the training data.
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        self.n_features_in_ = X.shape[1]
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

IMD_ORDER = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70-80", "80-90", "90-100"]
AGE_ORDER = ["0-35", "35-55", "55<="]
EDU_ORDER = ["No Formal quals", "Lower Than A Level", "A Level or Equivalent", "HE Qualification", "Post Graduate Qualification"]

ordinal_pipe = Pipeline([
    ("encode", OrdinalEncoder(categories=[IMD_ORDER, AGE_ORDER, EDU_ORDER],
                              handle_unknown="use_encoded_value", unknown_value=np.nan)),
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
nominal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
skewed_pipe = Pipeline([
    ("winsor", Winsorizer(0.01, 0.99)),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
other_num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("ordinal", ordinal_pipe, ORDINAL_COLS),
    ("nominal", nominal_pipe, NOMINAL_COLS),
    ("skewed",  skewed_pipe,  SKEWED_NUM),
    ("other",   other_num_pipe, OTHER_NUM),
], remainder="drop")

preprocessor

### 8.1 Check what the pipeline produces
We fit **on the training set only**, then transform both sets. We assert there are no missing values left and that the test set was not used for fitting.

In [ ]:
prep_check = Pipeline([("prep", preprocessor), ("var", VarianceThreshold(0.0))])
Xt_train = prep_check.fit_transform(X_train, y_train)
Xt_test = prep_check.transform(X_test)

feature_names = prep_check.named_steps["prep"].get_feature_names_out()
feature_names = feature_names[prep_check.named_steps["var"].get_support()]

assert not np.isnan(Xt_train).any() and not np.isnan(Xt_test).any(), "NaNs remain after preprocessing"
print("Transformed train:", Xt_train.shape, "| test:", Xt_test.shape, "| features kept:", len(feature_names))

# Winsorising limits and scaler means come from TRAIN only:
w = prep_check.named_steps["prep"].named_transformers_["skewed"].named_steps["winsor"]
print("\nExample learned 99th-percentile limit for total_clicks (train only):", round(w.hi_[0], 1))

In [ ]:
# Feature ranking on the TRAINING data only (mutual information with the target)
mi = mutual_info_classif(Xt_train, y_train, random_state=RANDOM_STATE)
mi_s = pd.Series(mi, index=feature_names).sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5))
mi_s[::-1].plot(kind="barh", color="#55A868")
plt.title("Top 15 features by mutual information (train only)")
plt.xlabel("Mutual information")
plt.tight_layout(); plt.show()

## 9. Sanity check: does the pipeline run end-to-end without leakage?
We attach two simple models to the pipeline and cross-validate **on the training set only** with student-grouped, stratified folds. In every fold the whole pipeline is re-fitted on the fold's training part, which is what makes this leakage-free. The test set is not touched. This is only a check of the pipeline, not our model comparison (that begins in Part 2).

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "f1": "f1"}

candidates = {
    "Dummy (stratified)": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
}

rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocessor), ("var", VarianceThreshold(0.0)), ("clf", clf)])
    res = cross_validate(pipe, X_train, y_train, groups=g_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"model": name, **{k: f"{res['test_' + k].mean():.3f} ± {res['test_' + k].std():.3f}" for k in scoring}})

pd.DataFrame(rows).set_index("model")

**Result note — fill in with your real numbers from the table above.** The logistic regression should beat the dummy baseline clearly on PR-AUC and F1 if the engineered features carry signal. If the scores look *suspiciously* perfect, suspect leakage and re-check the feature list in section 5.

## 10. Leakage audit (for the viva)
| Leakage risk | How this notebook prevents it |
|---|---|
| Target or outcome used as input | `final_result` and `date_unregistration` are excluded; an `assert` guards the feature list |
| Future information | VLE rows are filtered to `date <= CUTOFF_DAY` while reading; assessments to `date_submitted <= CUTOFF_DAY` |
| Students who already left | Removed from the cohort (they are not "at risk" at prediction time) |
| Preprocessing fitted on all data | Imputers, winsorising limits, scalers, encoders and the variance filter are inside the `Pipeline`, fitted on training folds only |
| Same student in train and test | `GroupShuffleSplit` and `StratifiedGroupKFold` split by `id_student` |
| Test set used for decisions | EDA, feature ranking and CV all use training data only |

## 11. Hand-off to Part 2
* The `preprocessor` object and the `X_train / X_test / y_train / y_test` split are reused unchanged in Part 2 (Naive Bayes vs decision tree). Note: Gaussian/Multinomial Naive Bayes may need a different encoding (e.g. non-negative features for Multinomial NB), so add a model-specific variant of the pipeline there rather than changing this one.
* Part 3 can compare cutoffs (30/60/90 days) using the same `CUTOFF_DAY` setting.

---
# Part 2 — Supervised Learning Baseline: Naive Bayes vs Decision Tree
**Course:** CS-13410, Fall 2026 | **Maps to:** CLO-3 | **Due:** 05-11-2026 (with Assignment 2)

**Objective:** apply and compare a probabilistic model (Naive Bayes) and a tree-based model (decision tree) on the project dataset built in Part 1.

| Brief requirement | Where in this section |
|---|---|
| Train and compare at least one Naive Bayes variant and a decision tree | 12.2 (Gaussian NB, Categorical NB, three decision trees) |
| One information-gain (or Gini) split computed by hand for a small worked case | 12.1 |
| Cross-validation and accuracy / F1 / ROC-AUC | 12.2 and 12.3 |
| Discuss which model performs better and why | 12.6 |

**Reused from Part 1 (run Part 1 first):** the cohort, the features, the student-grouped train/test split, the `preprocessor`, and the cross-validation splitter `cv`. The **test set stays untouched** here; held-out performance is reported in Part 3 after tuning.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.tree import DecisionTreeClassifier, plot_tree
import math

warnings.filterwarnings("ignore", category=UserWarning)

needed = ["X_train", "y_train", "g_train", "preprocessor", "cv", "FEATURES",
          "ORDINAL_COLS", "NOMINAL_COLS", "SKEWED_NUM", "OTHER_NUM", "IMD_ORDER", "AGE_ORDER", "EDU_ORDER"]
missing = [v for v in needed if v not in globals()]
assert not missing, f"Run the Part 1 cells first. Missing: {missing}"
print("Part 1 objects found. Training rows:", len(X_train))

## 12.1 Hand-worked split: information gain and Gini on a small sample
A decision tree chooses each split by how much it reduces **impurity**.

* **Entropy:**  H = − Σ pᵢ · log₂(pᵢ)  (used by ID3 / C4.5 and by scikit-learn with `criterion="entropy"`)
* **Gini impurity:**  G = 1 − Σ pᵢ²  (default of scikit-learn / CART)
* **Information gain** = H(parent) − [ (n_left / n) · H(left) + (n_right / n) · H(right) ]
* **Gini gain** is computed the same way using G.

To keep the calculation small enough to do on paper, we take **12 real students** from the training set (4 who withdrew and 8 who stayed, drawn at random with a fixed seed) and test a few simple yes/no splits. The code prints every number so you can copy the working into your write-up and re-check it by hand.

> The sample is deliberately small and its class mix is chosen for readability, so its gain values illustrate the method and are **not** a statement about the whole dataset (see the full-training-set check below).

In [ ]:
def counts(labels):
    return [int((labels == 0).sum()), int((labels == 1).sum())]

def entropy_expr(c):
    n = sum(c)
    terms = [f"({k}/{n})*log2({k}/{n})" for k in c if k > 0]
    val = -sum((k / n) * math.log2(k / n) for k in c if k > 0) + 0.0
    return "-" + " - ".join(terms) + f" = {val:.4f}", val

def gini_expr(c):
    n = sum(c)
    val = 1 - sum((k / n) ** 2 for k in c)
    return "1 - (" + " + ".join(f"({k}/{n})^2" for k in c) + f") = {val:.4f}", val

pos = X_train[y_train == 1].sample(4, random_state=RANDOM_STATE)
neg = X_train[y_train == 0].sample(8, random_state=RANDOM_STATE)
small = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_STATE)
small = small[["days_since_last_active", "active_days", "n_submitted"]].copy()
small["withdrew"] = y_train.loc[small.index].values
small = small.reset_index(drop=True)
small.index = small.index + 1
small.index.name = "student"
print("The 12 students used in the worked example:")
display(small)

In [ ]:
rules = {
    "days_since_last_active > 7":  lambda d: d["days_since_last_active"] > 7,
    "days_since_last_active > 14": lambda d: d["days_since_last_active"] > 14,
    "active_days <= 5":            lambda d: d["active_days"] <= 5,
    "n_submitted == 0":            lambda d: d["n_submitted"] == 0,
}

def gains(labels, mask):
    n = len(labels)
    left, right = labels[mask], labels[~mask]
    if len(left) == 0 or len(right) == 0:
        return None
    def g(f):
        return f(counts(labels)) - (len(left)/n)*f(counts(left)) - (len(right)/n)*f(counts(right))
    H = lambda c: entropy_expr(c)[1]
    G = lambda c: gini_expr(c)[1]
    return g(H), g(G)

rows = []
for name, rule in rules.items():
    g = gains(small["withdrew"], rule(small))
    if g is not None:
        rows.append({"candidate split": name, "information gain": round(g[0], 4), "Gini gain": round(g[1], 4)})
cand = pd.DataFrame(rows).sort_values("information gain", ascending=False).reset_index(drop=True)
print("All candidate splits on the 12 students:")
display(cand)

best_name = cand.loc[0, "candidate split"]
print(f"\nBest candidate by information gain: {best_name}")

In [ ]:
def show_split(name, rule, data):
    labels, mask = data["withdrew"], rule(data)
    n = len(labels)
    parent, left, right = counts(labels), counts(labels[mask]), counts(labels[~mask])
    print(f"SPLIT: {name}")
    print(f"  Parent : [stayed, withdrew] = {parent}  (n = {n})")
    print(f"  Left (condition TRUE)  : {left}  (n = {sum(left)})")
    print(f"  Right (condition FALSE): {right}  (n = {sum(right)})")
    for label, expr in [("ENTROPY / INFORMATION GAIN", entropy_expr), ("GINI", gini_expr)]:
        print(f"\n  --- {label} ---")
        ep, vp = expr(parent); el, vl = expr(left); er, vr = expr(right)
        print(f"  Parent impurity : {ep}")
        print(f"  Left impurity   : {el}")
        print(f"  Right impurity  : {er}")
        weighted = (sum(left)/n)*vl + (sum(right)/n)*vr
        print(f"  Weighted children = ({sum(left)}/{n})*{vl:.4f} + ({sum(right)}/{n})*{vr:.4f} = {weighted:.4f}")
        print(f"  GAIN = {vp:.4f} - {weighted:.4f} = {vp - weighted:.4f}")

show_split(best_name, rules[best_name], small)

In [ ]:
# Does the same rule behave similarly on the FULL training set? (context only)
full = X_train[["days_since_last_active", "active_days", "n_submitted"]].copy()
full["withdrew"] = y_train.values
mask_full = rules[best_name](full)
g_full = gains(full["withdrew"], mask_full)
print("Same rule on the full training set:")
print(f"  rows where rule is TRUE: {int(mask_full.sum()):,} | withdrawal rate TRUE: {full.loc[mask_full,'withdrew'].mean():.3f} | FALSE: {full.loc[~mask_full,'withdrew'].mean():.3f}")
print(f"  information gain = {g_full[0]:.4f} | Gini gain = {g_full[1]:.4f}")

**Write-up for 12.1.** The cell below drafts this write-up from the numbers computed above. Read it, check it against the printed working, and re-word it in your own words before submitting.

This is exactly what a tree does at every node: it tries many candidate splits, computes the gain for each, and keeps the best one. **ID3** picks the highest information gain (categorical features); **C4.5** uses gain ratio and handles numeric features; scikit-learn builds **CART** trees (binary splits with Gini or entropy).

In [ ]:
from IPython.display import Markdown
m_best = rules[best_name](small)
l_c, r_c = counts(small["withdrew"][m_best]), counts(small["withdrew"][~m_best])
ig_best = float(cand.loc[0, "information gain"]); gg_best = float(cand.loc[0, "Gini gain"])
gini_best = cand.loc[cand["Gini gain"].idxmax(), "candidate split"]
agree = "Entropy and Gini agree on the best split." if gini_best == best_name else f"Entropy prefers `{best_name}` but Gini prefers `{gini_best}`, so the two criteria disagree here."
rate_l = l_c[1] / max(sum(l_c), 1); rate_r = r_c[1] / max(sum(r_c), 1)
rt, rf = full.loc[mask_full, "withdrew"].mean(), full.loc[~mask_full, "withdrew"].mean()
same_dir = (rate_l > rate_r) == (rt > rf)
support = ("The same direction holds on the full training set, so the rule is not just an accident of the small sample."
           if same_dir else "On the full training set the direction is different, which shows that a 12-student sample can mislead.")
txt = f"""**Hand-worked split (draft).** We drew 12 real training students (4 who withdrew, 8 who stayed). The parent node has class counts [stayed, withdrew] = {counts(small['withdrew'])}. We tested {len(cand)} candidate yes/no splits. The best by information gain is `{best_name}`, with information gain {ig_best:.4f} and Gini gain {gg_best:.4f}. It sends {sum(l_c)} students to the TRUE branch ({l_c[1]} withdrew, {l_c[0]} stayed) and {sum(r_c)} to the FALSE branch ({r_c[1]} withdrew, {r_c[0]} stayed). {agree} The full calculation (entropy of the parent, entropy of each child, the weighted average, and the gain) is printed in the cell above.

On the full training set the same rule has a withdrawal rate of {rt:.3f} when TRUE and {rf:.3f} when FALSE (information gain {g_full[0]:.4f}, Gini gain {g_full[1]:.4f}). {support} A decision tree repeats exactly this procedure at every node and keeps the split with the highest gain."""
display(Markdown(txt))

## 12.2 Models and cross-validation
**Models compared (all evaluated inside a Pipeline, so nothing leaks):**

| Model | Preprocessing | Why |
|---|---|---|
| Dummy (stratified) | Part 1 pipeline | Floor that every model must beat |
| **Gaussian Naive Bayes** | Part 1 pipeline (made dense) | Assumes each feature is roughly normal given the class and independent of the others |
| **Categorical Naive Bayes** | Numeric features binned into 5 quantile bins, categories ordinal-encoded | Naive Bayes needs non-negative integer categories; scaled (negative) features do not work with it |
| Decision tree, unpruned (entropy) | Part 1 pipeline | Shows what happens when a tree grows without limits (overfitting) |
| Decision tree, depth 5 (entropy) | Part 1 pipeline | Controlled tree, information-gain criterion |
| Decision tree, depth 5 (Gini) | Part 1 pipeline | Same tree with the Gini criterion |

Trees do not need scaling, but reusing the Part 1 pipeline keeps the project consistent (monotone transforms such as log and standardisation do not change which splits a tree can make). Evaluation uses the same 5-fold `StratifiedGroupKFold` on the **training set only**.

In [ ]:
def make_dense(prep):
    p = clone(prep)
    p.set_params(sparse_threshold=0)
    return p

def to_int(a):
    return np.asarray(a).astype(int)

# Preprocessing specific to Categorical Naive Bayes
NB_CAT_COLS = NOMINAL_COLS + ["never_active"]
NB_NUM_COLS = SKEWED_NUM + [c for c in OTHER_NUM if c != "never_active"]

nb_ordinal = Pipeline([
    ("enc", OrdinalEncoder(categories=[IMD_ORDER, AGE_ORDER, EDU_ORDER],
                           handle_unknown="use_encoded_value", unknown_value=np.nan)),
    ("imp", SimpleImputer(strategy="most_frequent")),
])
nb_nominal = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)),
    ("imp2", SimpleImputer(strategy="most_frequent")),
])
nb_numeric = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("bin", KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")),
])
nb_prep = ColumnTransformer([
    ("ord", nb_ordinal, ORDINAL_COLS),
    ("nom", nb_nominal, NB_CAT_COLS),
    ("num", nb_numeric, NB_NUM_COLS),
])
N_NB_FEATURES = len(ORDINAL_COLS) + len(NB_CAT_COLS) + len(NB_NUM_COLS)

def tree_pipe(criterion, **kw):
    return Pipeline([
        ("prep", make_dense(preprocessor)),
        ("var", VarianceThreshold(0.0)),
        ("clf", DecisionTreeClassifier(criterion=criterion, class_weight="balanced",
                                       random_state=RANDOM_STATE, **kw)),
    ])

models = {
    "Dummy (stratified)": Pipeline([("prep", make_dense(preprocessor)), ("var", VarianceThreshold(0.0)),
                                    ("clf", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE))]),
    "Gaussian NB": Pipeline([("prep", make_dense(preprocessor)), ("var", VarianceThreshold(0.0)),
                             ("clf", GaussianNB())]),
    "Categorical NB (binned)": Pipeline([("prep", nb_prep), ("to_int", FunctionTransformer(to_int)),
                                         ("clf", CategoricalNB(min_categories=np.full(N_NB_FEATURES, 20)))]),
    "Tree unpruned (entropy)": tree_pipe("entropy"),
    "Tree depth 5 (entropy)": tree_pipe("entropy", max_depth=5, min_samples_leaf=50),
    "Tree depth 5 (gini)": tree_pipe("gini", max_depth=5, min_samples_leaf=50),
}
print("Models defined:", list(models))

In [ ]:
scoring = {"accuracy": "accuracy", "f1": "f1", "precision": "precision", "recall": "recall",
           "roc_auc": "roc_auc", "pr_auc": "average_precision"}

results, raw = [], {}
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, groups=g_train, cv=cv, scoring=scoring,
                         return_train_score=True, n_jobs=-1)
    raw[name] = res
    row = {"model": name}
    for m in scoring:
        row[m] = f"{res['test_' + m].mean():.3f} ± {res['test_' + m].std():.3f}"
    row["train ROC-AUC"] = f"{res['train_roc_auc'].mean():.3f}"
    results.append(row)

results_df = pd.DataFrame(results).set_index("model")
print("5-fold StratifiedGroupKFold on the training set (mean ± std over validation folds)")
results_df

In [ ]:
plot_df = pd.DataFrame({
    "ROC-AUC": {n: raw[n]["test_roc_auc"].mean() for n in raw},
    "PR-AUC":  {n: raw[n]["test_pr_auc"].mean() for n in raw},
})
ax = plot_df.plot(kind="barh", figsize=(9, 4.5), color=["#4C72B0", "#C44E52"])
ax.axvline(y_train.mean(), color="gray", ls="--", lw=1, label=f"PR-AUC of a random model ({y_train.mean():.2f})")
ax.set_title("Cross-validated ROC-AUC and PR-AUC (training set)")
ax.set_xlabel("score"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 12.3 ROC curves, precision-recall curves and confusion matrices
Using **out-of-fold** predicted probabilities (each training student is scored by a model that never saw that student's fold), we compare the main models. The 0.5 probability threshold is used for the confusion matrices; in a real Study-Sync reminder system the threshold would be chosen from the cost of a missed at-risk student versus a needless reminder.

In [ ]:
curve_models = ["Gaussian NB", "Categorical NB (binned)", "Tree depth 5 (entropy)", "Tree depth 5 (gini)"]
oof = {}
for name in curve_models:
    oof[name] = cross_val_predict(models[name], X_train, y_train, groups=g_train, cv=cv,
                                  method="predict_proba", n_jobs=-1)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in oof.items():
    RocCurveDisplay.from_predictions(y_train, p, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_train, p, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_title("ROC curves (out-of-fold)")
axes[1].axhline(y_train.mean(), color="gray", ls="--", lw=1); axes[1].set_title("Precision-recall curves (out-of-fold)")
plt.tight_layout(); plt.show()

for name, p in oof.items():
    cm = confusion_matrix(y_train, (p >= 0.5).astype(int))
    print(f"\n{name} — confusion matrix at threshold 0.5 (rows = true, columns = predicted)")
    display(pd.DataFrame(cm, index=["true stayed", "true withdrew"], columns=["pred stayed", "pred withdrew"]))

## 12.4 Tree depth: underfitting vs overfitting
An unrestricted tree can memorise the training data. We vary `max_depth` (all other settings default) and compare training and validation ROC-AUC with the same grouped cross-validation. Formal tuning of `max_depth`, `min_samples_leaf` and other hyperparameters is part of Part 3; this plot only explains why we restricted the tree above.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
tr, va = [], []
for d in depths:
    r = cross_validate(tree_pipe("entropy", max_depth=d), X_train, y_train, groups=g_train, cv=cv,
                       scoring="roc_auc", return_train_score=True, n_jobs=-1)
    tr.append(r["train_score"].mean()); va.append(r["test_score"].mean())

plt.figure(figsize=(8, 4.5))
plt.plot(depths, tr, "o-", label="training ROC-AUC")
plt.plot(depths, va, "s-", label="validation ROC-AUC")
plt.xlabel("max_depth"); plt.ylabel("ROC-AUC"); plt.title("Decision tree: training vs validation score by depth")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Depth with the best validation ROC-AUC:", depths[int(np.argmax(va))])

## 12.5 Looking inside the decision tree
We fit the depth-5 entropy tree on the **full training set** (no test data) to inspect its top splits and feature importances. Feature values in the tree are on the transformed scale (log-transformed and standardised), so thresholds are not in raw units.

In [ ]:
final_tree = clone(models["Tree depth 5 (entropy)"]).fit(X_train, y_train)
names = final_tree["prep"].get_feature_names_out()[final_tree["var"].get_support()]
tree = final_tree["clf"]

plt.figure(figsize=(18, 7))
plot_tree(tree, max_depth=3, feature_names=list(names), class_names=["stayed", "withdrew"],
          filled=True, rounded=True, fontsize=8)
plt.title("Top three levels of the depth-5 entropy tree")
plt.show()

imp = pd.Series(tree.feature_importances_, index=names).sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 4.5))
imp[::-1].plot(kind="barh", color="#55A868")
plt.title("Top 10 decision-tree feature importances"); plt.xlabel("importance")
plt.tight_layout(); plt.show()

## 12.6 Comparison and discussion
**The cell after this text drafts the section from your real results. Read it, check it against the tables and plots, and re-word it in your own words (about half a page).** Structure of the draft:

1. **Results.** Name the best Naive Bayes variant and the best decision tree, and quote their cross-validated ROC-AUC, PR-AUC and F1 against the dummy baseline. Say whether the differences are larger than the fold-to-fold standard deviation.
2. **Which performs better and why.** Use the points below only if your results support them.
   * *Naive Bayes assumes the features are conditionally independent given the class.* Our EDA showed strongly correlated engagement features (for example `total_clicks`, `clicks_last14` and `active_days`), so the independence assumption is violated and NB can double-count the same evidence.
   * *Gaussian NB assumes each feature is normally distributed within each class*, which fits one-hot columns poorly. Categorical NB avoids this by binning, but loses information inside each bin.
   * *A decision tree can model interactions and non-linear thresholds* (for example low recent activity combined with no submissions), which NB cannot.
   * *An unpruned tree overfits* (training ROC-AUC near 1 with a lower validation score, see 12.4); limiting depth and leaf size improves generalisation.
   * *Class imbalance:* at the 0.5 threshold, precision, recall and F1 depend on how each model handles the 15% minority class, so rely on ROC-AUC and PR-AUC for the comparison and report the confusion matrices honestly.
3. **Entropy vs Gini.** State whether the two criteria gave similar results (they usually do).
4. **Limitations.** Results are cross-validation scores on the training set only; no hyperparameter tuning yet; the test set is untouched.

In [ ]:
from IPython.display import Markdown

def ms(name, m):
    return raw[name]["test_" + m].mean(), raw[name]["test_" + m].std()

nb_names = ["Gaussian NB", "Categorical NB (binned)"]
tr_names = ["Tree depth 5 (entropy)", "Tree depth 5 (gini)"]
best_nb = max(nb_names, key=lambda n: ms(n, "roc_auc")[0])
best_tr = max(tr_names, key=lambda n: ms(n, "roc_auc")[0])
A = {m: ms(best_nb, m) for m in ["roc_auc", "pr_auc", "f1", "precision", "recall", "accuracy"]}
B = {m: ms(best_tr, m) for m in ["roc_auc", "pr_auc", "f1", "precision", "recall", "accuracy"]}
D = {m: ms("Dummy (stratified)", m) for m in ["roc_auc", "pr_auc", "f1"]}

gap = B["roc_auc"][0] - A["roc_auc"][0]
tol = max(A["roc_auc"][1], B["roc_auc"][1])
if abs(gap) <= tol:
    verdict = (f"The ROC-AUC difference between {best_nb} and {best_tr} ({gap:+.3f}) is not larger than the fold-to-fold "
               f"standard deviation ({tol:.3f}), so we cannot say that either family ranks students clearly better.")
elif gap > 0:
    verdict = (f"{best_tr} ranks students better than {best_nb} (ROC-AUC {B['roc_auc'][0]:.3f} vs {A['roc_auc'][0]:.3f}; "
               f"the gap of {gap:+.3f} is larger than the fold-to-fold standard deviation of {tol:.3f}).")
else:
    verdict = (f"{best_nb} ranks students better than {best_tr} (ROC-AUC {A['roc_auc'][0]:.3f} vs {B['roc_auc'][0]:.3f}; "
               f"the gap of {-gap:.3f} is larger than the fold-to-fold standard deviation of {tol:.3f}).")

f1_better = best_tr if B["f1"][0] > A["f1"][0] else best_nb
thr = (f"At the 0.5 threshold, {best_tr} has recall {B['recall'][0]:.3f} and precision {B['precision'][0]:.3f}, while {best_nb} has "
       f"recall {A['recall'][0]:.3f} and precision {A['precision'][0]:.3f}; {f1_better} has the higher F1 "
       f"({max(A['f1'][0], B['f1'][0]):.3f}). Because the classes are imbalanced, the 0.5 threshold is only one operating point, "
       f"which is why the threshold-free ROC-AUC and PR-AUC are the main comparison.")

un = "Tree unpruned (entropy)"
tr_un, va_un = raw[un]["train_roc_auc"].mean(), ms(un, "roc_auc")[0]
overfit = (f"The unpruned tree reaches a training ROC-AUC of {tr_un:.3f} but a validation ROC-AUC of only {va_un:.3f}, "
           f"a drop of {tr_un - va_un:.3f}, which is overfitting; limiting depth to 5 with at least 50 students per leaf raises the validation ROC-AUC to {B['roc_auc'][0]:.3f}."
           if tr_un - va_un > 0.05 else
           f"The unpruned tree has training ROC-AUC {tr_un:.3f} and validation ROC-AUC {va_un:.3f}, so overfitting is small here.")

cm_ = X_train[SKEWED_NUM].corr().abs().values.copy()
np.fill_diagonal(cm_, 0)
ii, jj = np.unravel_index(cm_.argmax(), cm_.shape)
corr_txt = (f"The engagement features are strongly correlated (for example `{SKEWED_NUM[ii]}` and `{SKEWED_NUM[jj]}`, |r| = {cm_[ii, jj]:.2f}). "
            f"Naive Bayes assumes the features are independent given the class, so correlated features are counted twice and its probabilities can be over-confident. "
            f"This violated assumption could limit Naive Bayes, but we have not tested it directly, and the scores above show whether it actually hurt.")

e_s, g_s = ms("Tree depth 5 (entropy)", "roc_auc")[0], ms("Tree depth 5 (gini)", "roc_auc")[0]
crit = (f"The entropy and Gini trees are almost identical (ROC-AUC {e_s:.3f} vs {g_s:.3f})." if abs(e_s - g_s) < 0.01
        else f"The entropy tree (ROC-AUC {e_s:.3f}) and the Gini tree ({g_s:.3f}) differ by {abs(e_s - g_s):.3f}.")

txt = f"""**Comparison of Naive Bayes and decision trees (draft, from our cross-validation results on the training set).**

**Results.** The stratified dummy baseline scores ROC-AUC {D['roc_auc'][0]:.3f} and PR-AUC {D['pr_auc'][0]:.3f} (about the {y_train.mean():.2f} share of withdrawing students), so every real model is judged against that floor. The best Naive Bayes variant is {best_nb} (ROC-AUC {A['roc_auc'][0]:.3f} ± {A['roc_auc'][1]:.3f}, PR-AUC {A['pr_auc'][0]:.3f}, F1 {A['f1'][0]:.3f}). The best decision tree is {best_tr} (ROC-AUC {B['roc_auc'][0]:.3f} ± {B['roc_auc'][1]:.3f}, PR-AUC {B['pr_auc'][0]:.3f}, F1 {B['f1'][0]:.3f}). {verdict}

**Threshold-dependent metrics.** {thr}

**Why the models behave this way.** {corr_txt} A decision tree makes no independence assumption and can combine thresholds on several features, for example low recent activity together with no submissions. {overfit} {crit}

**Limitations.** These are cross-validation scores on the training set only, with no hyperparameter tuning yet and the test set untouched. Part 3 will tune the models and report held-out test performance."""
display(Markdown(txt))

## 12.7 Hand-off to Part 3
* The test set remains unused. Part 3 will tune the models (grid / random search or Optuna) and report held-out test performance.
* Part 3 adds Random Forest, AdaBoost and a gradient-boosting model, plus Ridge, Lasso or Elastic Net where relevant, and the progress report.
* The depth-5 tree and the NB results above are the baseline to beat.